Gather updated lists of files, samples, metadata\
KL 2 April 2026\
Set up a database, just with sample information

In [556]:
%reset -f
#%whos #also useful at times

In [557]:
import pandas as pd
import os
import pdb

#need this to see the full column width
pd.set_option('display.max_colwidth', None)

In [558]:
# #now I see why Ben was deleting the database...otherwise get multiple inserts
# but I cannot get this to work as it is still in use and I am having trouble closing it.
# %tried;
# session.close()
# engine.dispose()
# def delete_db():
#     print('Deleting database')
#     db_path = 'new_database.db'
#     if os.path.exists(db_path):
# #         os.unlink(db_path)
#         os.remove(db_path)

In [559]:
from sqlalchemy import create_engine, Column, Integer, String
from sqlalchemy.orm import sessionmaker, declarative_base
from datetime import datetime

# create a SQLite database engine
#SQLALCHEMY_DATABASE_URL = "sqlite:///new_database.db"
#this will end up creating a new database everytime, but I need this for testing right now
SQLALCHEMY_DATABASE_URL = f"sqlite:///../test_data/new_database_{datetime.now().strftime('%Y%m%d_%H%M%S')}.db"
# delete_db()

#engine = create_engine(SQLALCHEMY_DATABASE_URL,echo=True)
engine = create_engine(SQLALCHEMY_DATABASE_URL)

# create a session factory
Session = sessionmaker(bind=engine)

# create a declarative base
Base = declarative_base()

In [560]:
# define the classes

class DiscreteInfo(Base):
    __tablename__ = 'discrete'
    id = Column(Integer, primary_key=True, index=True)
    bottleID = Column(String)
    cruise = Column(String)
    cast = Column(String)
    niskin = Column(String)
    nominalDepth = Column(String)
    V4data = Column(String)
    
    #latest, do I need the repr(self?)
    #need this next row to get the nice output (other get a generic thing ?: <__main__.DiscreteInfo object at 0x000001A3FC7A0F70>)
    def __repr__(self):
        return f"<DiscreteInfo(bottleID='{self.bottleID}', cruise='{self.cruise}', V4data='{self.V4data}')>"


class SeqInfoV1V2(Base):
    __tablename__ = 'sequencingV1V2'
    id = Column(Integer, primary_key=True, index=True)
    bottleID = Column(String)
    cast = Column(String)
    NominalDepth = Column(String)
    filename = Column(String)
    V1V2data = Column(String)
    #not sure how to do this next bit yet
    #casts = relationship('Cast', back_populates='cruise')
    
class SeqInfoV4(Base):
    __tablename__ = 'sequencingV4'
    id = Column(Integer, primary_key=True, index=True)
    bottleID = Column(String)
    cast = Column(String)
    NominalDepth = Column(String)
    filename = Column(String)
    V4data = Column(String)
    #not sure how to do this next bit yet
    #casts = relationship('Cast', back_populates='cruise')
    
    def __repr__(self) -> str:
        return f"User(id={self.id!r}, name={self.bottleID!r}, V4data={self.V4data!r})"

    
class CyverseInfo(Base):
    __tablename__ = 'cyverse'
    id = Column(Integer, primary_key=True, index=True)
    filename = Column(String)  
    def __repr__(self) -> str:
        return f"index(id={self.id!r}, filename={self.filename!r})"

In [561]:
# create the database tables
Base.metadata.create_all(engine)

In [562]:
# # insert some data, setup functions, one per data type
def load_discrete_info():
    print('Loading discrete sample information')
    data_dir = '../test_data/BIOS-SCOPE time series/'
    fName = 'BATS_BS_COMBINED_MASTER_mini.xlsx'
    df = pd.DataFrame(pd.read_excel(os.path.join(data_dir,fName),sheet_name='DATA'))

    session = Session()
    for index, row in df.iterrows():
        #pdb.set_trace()
        db = DiscreteInfo()
        db.bottleID = row['New_ID'] 
        db.cruise = row['Cruise_ID']
        db.cast = row['Cast']
        db.niskin = row['Niskin']
        db.nominalDepth = row['Nominal_Depth']
        session.add(db)
    
    session.commit()
    
def load_V4_sequencing_info():
    print('Loading V4 sequencing information')
    data_dir = '../test_data/BIOS-SCOPE time series/'
    fName = 'V4_dada2_read_info_03052026.xlsx'
    df = pd.DataFrame(pd.read_excel(os.path.join(data_dir,fName)))
    #strip the @#%@#^$ spaces in headers
    df.columns = df.columns.str.replace(' ','') ## actual file has a space AFTER Bottle ID !

    #df = pd.read_csv('test_data/cruise_data.csv')
    session = Session()
    for index, row in df.iterrows():
        #pdb.set_trace()
        db = SeqInfoV4()
        db.bottleID = row['BottleID'] 
        db.cast = row['Cast']
        db.filename = row['FilenameinCyverse']
        db.V4data = fName
        session.add(db)
    
    session.commit()
    
def load_V1V2_sequencing_info():
    print('Loading V1V2 sequencing information')
    data_dir = '../test_data/BIOS-SCOPE time series/'
    fName = 'V1V2_dada2_read_info_03052026.xlsx'
    df = pd.DataFrame(pd.read_excel(os.path.join(data_dir,fName)))
    #strip the @#%@#^$ spaces in headers
    df.columns = df.columns.str.replace(' ','') ## actual file has a space AFTER Bottle ID !

    #df = pd.read_csv('test_data/cruise_data.csv')
    session = Session()
    for index, row in df.iterrows():
        #pdb.set_trace()
        db = SeqInfoV1V2()
        db.bottleID = row['BottleID'] 
        db.cast = row['Cast']
        db.filename = row['FileName']
        db.V1V2data = fName
        session.add(db)
    
    session.commit()

def load_cyverse_info():
    print('Loading sequencing information')
    dataDir = '../test_data/BIOS-SCOPE time series/'
    fName = 'files_shortList.txt'
    df = pd.read_csv(os.path.join(dataDir,fName),sep='\t',header=None,comment = '#')

    #strip off the end of the filename
    for index,row in df.iterrows():
        #file = os.path.basename(row.to_string()).strip('fastq.gz')
        file = os.path.basename(row.to_string()).strip('.gz')
        df.loc[index,'filename'] = file

    session = Session()
    for index, row in df.iterrows():
        db = CyverseInfo()
        db.filename = row['filename'] 
        session.add(db)
    
    session.commit()

In [563]:
#now run the functions
load_V4_sequencing_info()
load_V1V2_sequencing_info()
load_cyverse_info()
load_discrete_info()

Loading V4 sequencing information
Loading V1V2 sequencing information
Loading sequencing information
Loading discrete sample information


In [564]:
from sqlalchemy import inspect
inspector = inspect(engine)
print(inspector.get_table_names())

['cyverse', 'discrete', 'sequencingV1V2', 'sequencingV4']


In [565]:
from sqlalchemy import create_engine, inspect, MetaData, Table
metadata_obj = MetaData()
metadata_obj.reflect(bind=engine)
#print(f"Tables found: {metadata_obj.tables.keys()}") #

#mmm I think these lines put the data into a form I can use here but are NOT altering the existing database
# which is great, until I want to update the existing database
user_seqV4 = Table('sequencingV4', metadata_obj, autoload_with=engine)
user_seqV1V2 = Table('sequencingV1V2', metadata_obj, autoload_with=engine)
user_cy = Table('cyverse',metadata_obj,autoload_with=engine)
user_discrete = Table('discrete',metadata_obj,autoload_with=engine)

user_discrete

Table('discrete', MetaData(), Column('id', INTEGER(), table=<discrete>, primary_key=True, nullable=False), Column('bottleID', VARCHAR(), table=<discrete>), Column('cruise', VARCHAR(), table=<discrete>), Column('cast', VARCHAR(), table=<discrete>), Column('niskin', VARCHAR(), table=<discrete>), Column('nominalDepth', VARCHAR(), table=<discrete>), Column('V4data', VARCHAR(), table=<discrete>), schema=None)

In [566]:
user_seqV4

Table('sequencingV4', MetaData(), Column('id', INTEGER(), table=<sequencingV4>, primary_key=True, nullable=False), Column('bottleID', VARCHAR(), table=<sequencingV4>), Column('cast', VARCHAR(), table=<sequencingV4>), Column('NominalDepth', VARCHAR(), table=<sequencingV4>), Column('filename', VARCHAR(), table=<sequencingV4>), Column('V4data', VARCHAR(), table=<sequencingV4>), schema=None)

In [567]:
from sqlalchemy import select
from sqlalchemy.orm import Session

stmt = select(
    user_seqV4.c.bottleID, 
    user_cy.c.filename
 ).join_from(
    user_seqV4,
    user_cy,
    user_cy.c.filename == user_seqV4.c.filename
)

#session.scalars(stmt).one()
with Session(engine) as session:
    for row in session.execute(stmt):
        print(row)

('1035501701', 'lane1-s001-indexN716-D-S518-D-ACTCGCTA-CTATTAAG-BSv4-342_S1_L001_R1_001.fastq')
('1035501703', 'lane1-s002-indexN716-D-S520-D-ACTCGCTA-AAGGCTAT-BSv4-343_S2_L001_R1_001.fastq')
('1035501705', 'lane1-s003-indexN716-D-S521-D-ACTCGCTA-GAGCCTTA-BSv4-344_S3_L001_R1_001.fastq')
('1035501707', 'lane1-s004-indexN716-D-S522-D-ACTCGCTA-TTATGCGA-BSv4-345_S4_L001_R1_001.fastq')
('1035501709', 'lane1-s005-indexN716-D-S513-D-ACTCGCTA-TCGACTAG-BSv4-346_S5_L001_R1_001.fastq')
('1035501719', 'lane1-s010-indexN718-D-S520-D-GGAGCTAC-AAGGCTAT-BSv4-351_S10_L001_R1_001.fastq')
('1035501721', 'lane1-s011-indexN718-D-S521-D-GGAGCTAC-GAGCCTTA-BSv4-352_S11_L001_R1_001.fastq')
('1035501723', 'lane1-s012-indexN718-D-S522-D-GGAGCTAC-TTATGCGA-BSv4-353_S12_L001_R1_001.fastq')
('1035601501', 'lane1-s013-indexN718-D-S513-D-GGAGCTAC-TCGACTAG-BSv4-354_S13_L001_R1_001.fastq')
('1035601503', 'lane1-s014-indexN718-D-S515-D-GGAGCTAC-TTCTAGCT-BSv4-355_S14_L001_R1_001.fastq')
('1035601505', 'lane1-s015-indexN71

### new path from here...don't create a dataframe (yet)...update the existing database

In [568]:
session.query(SeqInfoV4).all()

[User(id=1, name='1035501701', V4data='V4_dada2_read_info_03052026.xlsx'),
 User(id=2, name='1035501703', V4data='V4_dada2_read_info_03052026.xlsx'),
 User(id=3, name='1035501705', V4data='V4_dada2_read_info_03052026.xlsx'),
 User(id=4, name='1035501707', V4data='V4_dada2_read_info_03052026.xlsx'),
 User(id=5, name='1035501709', V4data='V4_dada2_read_info_03052026.xlsx'),
 User(id=6, name='1035501711', V4data='V4_dada2_read_info_03052026.xlsx'),
 User(id=7, name='1035501713', V4data='V4_dada2_read_info_03052026.xlsx'),
 User(id=8, name='1035501715', V4data='V4_dada2_read_info_03052026.xlsx'),
 User(id=9, name='1035501717', V4data='V4_dada2_read_info_03052026.xlsx'),
 User(id=10, name='1035501719', V4data='V4_dada2_read_info_03052026.xlsx'),
 User(id=11, name='1035501721', V4data='V4_dada2_read_info_03052026.xlsx'),
 User(id=12, name='1035501723', V4data='V4_dada2_read_info_03052026.xlsx'),
 User(id=13, name='1035601501', V4data='V4_dada2_read_info_03052026.xlsx'),
 User(id=14, name='10

In [569]:
user_seqV4

Table('sequencingV4', MetaData(), Column('id', INTEGER(), table=<sequencingV4>, primary_key=True, nullable=False), Column('bottleID', VARCHAR(), table=<sequencingV4>), Column('cast', VARCHAR(), table=<sequencingV4>), Column('NominalDepth', VARCHAR(), table=<sequencingV4>), Column('filename', VARCHAR(), table=<sequencingV4>), Column('V4data', VARCHAR(), table=<sequencingV4>), schema=None)

In [570]:
#updating...
from sqlalchemy import update,select

# 1. Define the subquery to fetch a value from the second table
scalar_subq = (
    select(user_seqV4.c.V4data)
    .where(user_discrete.c.bottleID == user_seqV4.c.bottleID)
    .limit(1)
    .scalar_subquery()
)

# 2. Use the subquery in the .values() clause of an update statement
stmt = update(user_discrete).values(V4data=scalar_subq)


#then execute the statement
with engine.connect() as conn:
    result = conn.execute(stmt)
    conn.commit()
    

In [571]:
# see if this worked
Base.metadata.create_all(engine)

# Create a session
Session = sessionmaker(bind=engine)
session = Session()

from sqlalchemy import Table, Column, Integer, String, MetaData

metadata = MetaData()
users = Table('discrete', metadata,
    Column('id', Integer, primary_key=True),
    Column('bottleID', String),
    Column('cruise', String),
    Column('V4data',String)
)

#how to execute a query
stmt = select(users)
with engine.connect() as conn:
    result = conn.execute(stmt)
    for row in result:
        print(row)

(1, '1033900707', 'AE1718', None)
(2, '1033900708', 'AE1718', None)
(3, '1033900709', 'AE1718', None)
(4, '1033900710', 'AE1718', None)
(5, '1033900711', 'AE1718', None)
(6, '1033900712', 'AE1718', None)
(7, '1033900713', 'AE1718', None)
(8, '1033900714', 'AE1718', None)
(9, '1033900715', 'AE1718', None)
(10, '1033900716', 'AE1718', None)
(11, '1033900717', 'AE1718', None)
(12, '1033900718', 'AE1718', None)
(13, '1033900719', 'AE1718', None)
(14, '1033900720', 'AE1718', None)
(15, '1033900721', 'AE1718', None)
(16, '1033900722', 'AE1718', None)
(17, '1033900723', 'AE1718', None)
(18, '1033900724', 'AE1718', None)
(19, '1033900801', 'AE1718', None)
(20, '1033900802', 'AE1718', None)
(21, '1033900803', 'AE1718', None)
(22, '1033900804', 'AE1718', None)
(23, '1033900805', 'AE1718', None)
(24, '1033900806', 'AE1718', None)
(25, '1033900807', 'AE1718', None)
(26, '1033900808', 'AE1718', None)
(27, '1033900809', 'AE1718', None)
(28, '1033900810', 'AE1718', None)
(29, '1033900811', 'AE1718', 

In [503]:
user_discrete

Table('discrete', MetaData(), Column('id', INTEGER(), table=<discrete>, primary_key=True, nullable=False), Column('bottleID', VARCHAR(), table=<discrete>), Column('cruise', VARCHAR(), table=<discrete>), Column('cast', VARCHAR(), table=<discrete>), Column('niskin', VARCHAR(), table=<discrete>), Column('nominalDepth', VARCHAR(), table=<discrete>), schema=None)

In [496]:
print(stmt)

UPDATE discrete SET cruise=(SELECT discrete."bottleID" AS "New_ID" 
FROM discrete LEFT OUTER JOIN "sequencingV4" ON discrete."bottleID" = "sequencingV4"."bottleID")


In [499]:
# Create all tables in the engine
Base.metadata.create_all(engine)

# Create a session
Session = sessionmaker(bind=engine)
session = Session()

from sqlalchemy import Table, Column, Integer, String, MetaData

metadata = MetaData()
users = Table('discrete', metadata,
    Column('id', Integer, primary_key=True),
    Column('bottleID', String),
    Column('cruise', String),
    #Column('V4data',String)
)

#how to execute a query
stmt = select(users)
with engine.connect() as conn:
    result = conn.execute(stmt)
    for row in result:
        print(row)

(1, '1033900707', '1033900707')
(2, '1033900708', '1033900707')
(3, '1033900709', '1033900707')
(4, '1033900710', '1033900707')
(5, '1033900711', '1033900707')
(6, '1033900712', '1033900707')
(7, '1033900713', '1033900707')
(8, '1033900714', '1033900707')
(9, '1033900715', '1033900707')
(10, '1033900716', '1033900707')
(11, '1033900717', '1033900707')
(12, '1033900718', '1033900707')
(13, '1033900719', '1033900707')
(14, '1033900720', '1033900707')
(15, '1033900721', '1033900707')
(16, '1033900722', '1033900707')
(17, '1033900723', '1033900707')
(18, '1033900724', '1033900707')
(19, '1033900801', '1033900707')
(20, '1033900802', '1033900707')
(21, '1033900803', '1033900707')
(22, '1033900804', '1033900707')
(23, '1033900805', '1033900707')
(24, '1033900806', '1033900707')
(25, '1033900807', '1033900707')
(26, '1033900808', '1033900707')
(27, '1033900809', '1033900707')
(28, '1033900810', '1033900707')
(29, '1033900811', '1033900707')
(30, '1033900812', '1033900707')
(31, '1033900813', 

In [500]:
#now, with the discrete data, find the rows there with matching Bottle ID in the seqInfo file
#set this up as a left outer join (all rows of discrete and only those rows of seqdata that match)
#start tidying this up to make it useful. Plan is to ultimately send out one table with all
#the discrete information and columns for cases where there is V1V2, V4, mtabs...

stmt = select(
    user_discrete.c.bottleID.label("New_ID"), 
    user_discrete,
    user_seqV4.c.V4data
).join_from(
    user_discrete,
    user_seqV4,
    user_discrete.c.bottleID == user_seqV4.c.bottleID,
    isouter=True
)

#session.scalars(stmt).one()
with Session(engine) as session:
#     for row in session.execute(stmt):
#         print(row)
    rows = session.execute(stmt).all()
    table_data = [row._mapping for row in rows]
    df = pd.DataFrame(table_data)

TypeError: __call__() takes 1 positional argument but 2 were given

In [363]:
df.head()

,New_ID,V4data,bottleID,cast,cruise,id,niskin,nominalDepth
0,1033900707,None,1033900707,7,AE1718,1,7,40
1,1033900708,None,1033900708,7,AE1718,2,8,40
2,1033900709,None,1033900709,7,AE1718,3,9,60
3,1033900710,None,1033900710,7,AE1718,4,10,60
4,1033900711,None,1033900711,7,AE1718,5,11,80


In [365]:
df.to_csv('tempb4.csv')

In [366]:
#now let's see if I can get this to modify the database as I go

In [383]:
df.iloc[0,]

New_ID          1033900707
V4data                None
bottleID        1033900707
cast                     7
cruise              AE1718
id                       1
niskin                   7
nominalDepth            40
Name: 0, dtype: object

In [373]:
df.iloc[43,]

New_ID                                1033901102
V4data          V4_dada2_read_info_03052026.xlsx
bottleID                              1033901102
cast                                          11
cruise                                    AE1718
id                                            44
niskin                                         2
nominalDepth                                   1
Name: 43, dtype: object

In [385]:
SeqInfoV4

__main__.SeqInfoV4

In [398]:
from sqlalchemy import update, bindparam
# stmt = (
#     update(user_table)
#     .where(user_table.c.name == "patrick")
#     .values(fullname="Patrick the Star")
# )
# print(stmt)

stmt = (
    update(user_discrete)
    .where(user_discrete.c.bottleID == bindparam("1033900708"))
    .values(cruise=bindparam("test"))
)
print(stmt)

UPDATE discrete SET cruise=:test WHERE discrete."bottleID" = :1033900708


In [411]:
from sqlalchemy import func
subq = (
    select(func.count(user_discrete.c.id))
    .where(user_discrete.c.bottleID == user_seqV4.c.bottleID)
    .scalar_subquery()
    .correlate(user_discrete)
)
print(subq)


(SELECT count(discrete.id) AS count_1 
FROM discrete, "sequencingV4" 
WHERE discrete."bottleID" = "sequencingV4"."bottleID")


In [412]:
with engine.connect() as conn:
    result = conn.execute(
        select(
            user_discrete.c.bottleID,
            user_seqV4.c.bottleID,
            subq.label("address_count"), #not even sure what this is but it will be third column
        )
        .join_from(
            user_discrete,
            user_seqV4,
            user_discrete.c.bottleID == user_seqV4.c.bottleID)
    )
    print(result.all())
    

[('1033901102', '1033901102', 286)]


In [423]:
scalar_subq = (
    select(user_discrete.c.id)
    .where(user_discrete.c.bottleID == "1033900707")
    .limit(1)
    .scalar_subquery() #must be at the end
)
update_stmt = update(user_discrete).values(bottleID = scalar_subq)
print(update_stmt)

UPDATE discrete SET "bottleID"=(SELECT discrete.id 
FROM discrete 
WHERE discrete."bottleID" = :bottleID_1
 LIMIT :param_1)


In [424]:
from sqlalchemy import select

with engine.connect() as connection:
    query = select(user_discrete)
    result = connection.execute(query)
    for row in result:
        print(row) # Access values via row.column_name or index

(1, '1033900707', 'AE1718', '7', '7', '40')
(2, '1033900708', 'AE1718', '7', '8', '40')
(3, '1033900709', 'AE1718', '7', '9', '60')
(4, '1033900710', 'AE1718', '7', '10', '60')
(5, '1033900711', 'AE1718', '7', '11', '80')
(6, '1033900712', 'AE1718', '7', '12', '80')
(7, '1033900713', 'AE1718', '7', '13', '100')
(8, '1033900714', 'AE1718', '7', '14', '100')
(9, '1033900715', 'AE1718', '7', '15', '120')
(10, '1033900716', 'AE1718', '7', '16', '120')
(11, '1033900717', 'AE1718', '7', '17', '140')
(12, '1033900718', 'AE1718', '7', '18', '140')
(13, '1033900719', 'AE1718', '7', '19', '160')
(14, '1033900720', 'AE1718', '7', '20', '160')
(15, '1033900721', 'AE1718', '7', '21', '200')
(16, '1033900722', 'AE1718', '7', '22', '200')
(17, '1033900723', 'AE1718', '7', '23', '250')
(18, '1033900724', 'AE1718', '7', '24', '250')
(19, '1033900801', 'AE1718', '8', '1', '1')
(20, '1033900802', 'AE1718', '8', '2', '10')
(21, '1033900803', 'AE1718', '8', '3', '20')
(22, '1033900804', 'AE1718', '8', '4',

In [425]:
session.flush()

In [ ]:
#apparentely a new way to do this (though the old way is still supported)

In [465]:
%reset -f

In [466]:
from sqlalchemy import create_engine, Column, Integer, String
from sqlalchemy.orm import sessionmaker, declarative_base
from datetime import datetime

# create a SQLite database engine
#SQLALCHEMY_DATABASE_URL = "sqlite:///new_database.db"
#this will end up creating a new database everytime, but I need this for testing right now
SQLALCHEMY_DATABASE_URL = f"sqlite:///../test_data/new_database_{datetime.now().strftime('%Y%m%d_%H%M%S')}.db"
# delete_db()

#engine = create_engine(SQLALCHEMY_DATABASE_URL,echo=True)
engine = create_engine(SQLALCHEMY_DATABASE_URL)

In [467]:
from sqlalchemy.orm import DeclarativeBase
class Base(DeclarativeBase):
    pass

In [468]:
Base.metadata

MetaData()

In [469]:
from typing import List
from typing import Optional
from sqlalchemy import Column, String
from sqlalchemy.orm import Mapped
from sqlalchemy.orm import mapped_column
from sqlalchemy.orm import relationship

class TestingNew(Base):
    __tablename__ = 'testingNew'
    id: Mapped[int] = mapped_column(primary_key=True)
    bottleID: Mapped[int] = mapped_column(String(10))
    cruise: Mapped[str] = mapped_column(String(10))
    cast: Mapped[int] = mapped_column(String(10))
    niskin: Mapped[int] = mapped_column(String(10))
    nominalDepth: Mapped[int] = mapped_column(String(10))
    
# class User(Base):
#     __tablename__ = "user_account"
#     id: Mapped[int] = mapped_column(primary_key=True)
#     name: Mapped[str] = mapped_column(String(30))
#     fullname: Mapped[Optional[str]]
#     addresses: Mapped[List["Address"]] = relationship(back_populates="user")
#     def __repr__(self) -> str:
#         return f"User(id={self.id!r}, name={self.name!r}, fullname={self.fullname!r})"

# class Address(Base):
#     __tablename__ = "address"
#     id: Mapped[int] = mapped_column(primary_key=True)
#     email_address: Mapped[str]
#     user_id = mapped_column(ForeignKey("user_account.id"))
#     user: Mapped[User] = relationship(back_populates="addresses")
#     def __repr__(self) -> str:
#         return f"Address(id={self.id!r}, email_address={self.email_address!r})"

In [470]:
Base.metadata.create_all(engine)

In [472]:
rat = TestingNew(bottleID='test',cruise = 'test')

In [473]:
rat

In [427]:
session = Session(engine)

In [428]:
session.add(rat)

In [429]:
session.new

IdentitySet([<__main__.DiscreteInfo object at 0x000001F93749F850>])

In [368]:
for index,row in df.iterrows():
    print(row)

New_ID          1033900707
V4data                None
bottleID        1033900707
cast                     7
cruise              AE1718
id                       1
niskin                   7
nominalDepth            40
Name: 0, dtype: object
New_ID          1033900708
V4data                None
bottleID        1033900708
cast                     7
cruise              AE1718
id                       2
niskin                   8
nominalDepth            40
Name: 1, dtype: object
New_ID          1033900709
V4data                None
bottleID        1033900709
cast                     7
cruise              AE1718
id                       3
niskin                   9
nominalDepth            60
Name: 2, dtype: object
New_ID          1033900710
V4data                None
bottleID        1033900710
cast                     7
cruise              AE1718
id                       4
niskin                  10
nominalDepth            60
Name: 3, dtype: object
New_ID          1033900711
V4data           

New_ID          1034800611
V4data                None
bottleID        1034800611
cast                     6
cruise              AE1817
id                     420
niskin                  11
nominalDepth           400
Name: 439, dtype: object
New_ID          1034800612
V4data                None
bottleID        1034800612
cast                     6
cruise              AE1817
id                     421
niskin                  12
nominalDepth           500
Name: 440, dtype: object
New_ID          1034800613
V4data                None
bottleID        1034800613
cast                     6
cruise              AE1817
id                     422
niskin                  13
nominalDepth           600
Name: 441, dtype: object
New_ID          1034800614
V4data                None
bottleID        1034800614
cast                     6
cruise              AE1817
id                     423
niskin                  14
nominalDepth          1000
Name: 442, dtype: object
New_ID          1034800615
V4data   

New_ID          1035100817
V4data                None
bottleID        1035100817
cast                     8
cruise              AE1825
id                     831
niskin                  17
nominalDepth           200
Name: 874, dtype: object
New_ID          1035100818
V4data                None
bottleID        1035100818
cast                     8
cruise              AE1825
id                     832
niskin                  18
nominalDepth           200
Name: 875, dtype: object
New_ID          1035100819
V4data                None
bottleID        1035100819
cast                     8
cruise              AE1825
id                     833
niskin                  19
nominalDepth           300
Name: 876, dtype: object
New_ID          1035100820
V4data                None
bottleID        1035100820
cast                     8
cruise              AE1825
id                     834
niskin                  20
nominalDepth           300
Name: 877, dtype: object
New_ID          1035100821
V4data   

New_ID          1035401220
V4data                None
bottleID        1035401220
cast                    12
cruise              AE1836
id                    1209
niskin                  20
nominalDepth          4000
Name: 1277, dtype: object
New_ID          1035401221
V4data                None
bottleID        1035401221
cast                    12
cruise              AE1836
id                    1210
niskin                  21
nominalDepth          4200
Name: 1278, dtype: object
New_ID          1035401222
V4data                None
bottleID        1035401222
cast                    12
cruise              AE1836
id                    1211
niskin                  22
nominalDepth          4530
Name: 1279, dtype: object
New_ID          1035401223
V4data                None
bottleID        1035401223
cast                    12
cruise              AE1836
id                    1212
niskin                  23
nominalDepth          4530
Name: 1280, dtype: object
New_ID          1035401224
V4dat

New_ID          1035701809
V4data                None
bottleID        1035701809
cast                    18
cruise               EN631
id                    1684
niskin                   9
nominalDepth           200
Name: 1760, dtype: object
New_ID          1035701810
V4data                None
bottleID        1035701810
cast                    18
cruise               EN631
id                    1685
niskin                  10
nominalDepth           300
Name: 1761, dtype: object
New_ID          1035701811
V4data                None
bottleID        1035701811
cast                    18
cruise               EN631
id                    1686
niskin                  11
nominalDepth           400
Name: 1762, dtype: object
New_ID          1035701812
V4data                None
bottleID        1035701812
cast                    18
cruise               EN631
id                    1687
niskin                  12
nominalDepth           500
Name: 1763, dtype: object
New_ID          1035701813
V4dat

New_ID          1036101511
V4data                None
bottleID        1036101511
cast                    15
cruise              AE1917
id                    2105
niskin                  11
nominalDepth           100
Name: 2181, dtype: object
New_ID          1036101512
V4data                None
bottleID        1036101512
cast                    15
cruise              AE1917
id                    2106
niskin                  12
nominalDepth           100
Name: 2182, dtype: object
New_ID          1036101513
V4data                None
bottleID        1036101513
cast                    15
cruise              AE1917
id                    2107
niskin                  13
nominalDepth           120
Name: 2183, dtype: object
New_ID          1036101514
V4data                None
bottleID        1036101514
cast                    15
cruise              AE1917
id                    2108
niskin                  14
nominalDepth           120
Name: 2184, dtype: object
New_ID          1036101515
V4dat

In [ ]:
#now, how do I insert this information back into discrete (don't really want to create a new table)

In [ ]:
type(user_discrete)

In [ ]:
from sqlalchemy import update
stmt = (
    update(user_discrete)
    .where(user_table.c.name == "patrick")
    .values(fullname="Patrick the Star")
)
print(stmt)


In [ ]:
SQLALCHEMY_DATABASE_URL

In [ ]:
stmt = select(user_discrete)
with Session(engine) as session:
    session.execute(stmt).all()
    session.commit()

In [ ]:
type(rows)

In [ ]:
sandy = session.execute(select(user_discrete).filter_by(user_discrete.c.getDataHere == "V4_dada2_read_info_03052026.xlsx")).scalar_one()

In [ ]:
stmt = select(
    user_discrete.c.bottleID.label("New_ID"), 
    user_discrete,
    user_seq.c.getDataHere
).join_from(
    user_discrete,
    user_seq,
    user_discrete.c.bottleID == user_seq.c.bottleID,
    isouter=True
)

#session.scalars(stmt).one()
with Session(engine) as session:
#     for row in session.execute(stmt):
#         print(row)
    rows = session.execute(stmt).all()
    session.add(rows)
#     table_data = [row._mapping for row in rows]
#     df = pd.DataFrame(table_data)

In [ ]:
from sqlalchemy import select,insert

select_stmt = select(
    user_discrete.c.bottleID.label("New_ID"), 
    user_discrete,
    user_seq.c.getDataHere
).join_from(
    user_discrete,
    user_seq,
    user_discrete.c.bottleID == user_seq.c.bottleID,
    isouter=True
)

insert_stmt = insert(user_discrete).from_select(
    ["id","getDataHere"],select_stmt)

# select_stmt = select(user_table.c.id, user_table.c.name + "@aol.com")
# insert_stmt = insert(address_table).from_select(
#     ["user_id", "email_address"], select_stmt
# )
print(select_stmt) #OK
print(insert_stmt) #fails: key error

In [ ]:
print(insert_stmt)

In [ ]:
df.to_csv('temp2.csv',index=False)

In [ ]:
session.close()

In [ ]:
#Stick some code below this spot as a holding zone

raise SystemExit("Stop execution here")

In [ ]:
from sqlalchemy import Boolean, Column, ForeignKey, Integer, String, Date, Float, DateTime
from sqlalchemy.orm import relationship

#moved the database.py file into the 'notebooks' folder, so this next line reads database.py
#from database import Base



In [ ]:
from sqlalchemy import create_engine

SQLALCHEMY_DATABASE_URL = "sqlite:///new_database.db"

engine = create_engine(SQLALCHEMY_DATABASE_URL,echo=True)

In [ ]:
from sqlalchemy import MetaData
metadata_obj = MetaData()

In [ ]:
from sqlalchemy import Table, Column, Integer, String
user_table_seqInfo = Table(
    "seqInfo",
    metadata_obj,
    Column("id", Integer, primary_key=True),
    Column("bottleID", String(30)),
    Column("cast", String),
    Column("NominalDepth", String),
    Column("filename", String),
)

In [ ]:
user_table_seqInfo.primary_key

In [ ]:
metadata_obj.create_all(engine)

In [ ]:
#note sure I udnerstand this, but enter for now
from sqlalchemy.orm import DeclarativeBase
class Base(DeclarativeBase):
    pass

In [ ]:
Base.metadata

In [ ]:
##now we want to declare our classes
from typing import List
from typing import Optional
from sqlalchemy.orm import Mapped
from sqlalchemy.orm import mapped_column
from sqlalchemy.orm import relationship

# class User(Base):
#     __tablename__ = "user_account"
#     id: Mapped[int] = mapped_column(primary_key=True)
#     name: Mapped[str] = mapped_column(String(30))
#     fullname: Mapped[Optional[str]]
#     addresses: Mapped[List["Address"]] = relationship(back_populates="user")
#     def __repr__(self) -> str:
#         return f"User(id={self.id!r}, name={self.name!r}, fullname={self.fullname!r})"

# class Address(Base):
#     __tablename__ = "address"
#     id: Mapped[int] = mapped_column(primary_key=True)
#     email_address: Mapped[str]
#     user_id = mapped_column(ForeignKey("user_account.id"))
#     user: Mapped[User] = relationship(back_populates="addresses")
#     def __repr__(self) -> str:
#         return f"Address(id={self.id!r}, email_address={self.email_address!r})"

#mote difference in syntax from example...this is new in SQLAlchemy 1.4

class SeqInfo(Base):
    __tablename__ = 'sequencingInfo'
    id = Column(Integer, primary_key=True, index=True)
    bottleID = Column(String)
    cast = Column(String)
    NominalDepth = Column(String)
    filename = Column(String)
    #not sure how to do this next bit yet
    #casts = relationship('Cast', back_populates='cruise')
    
    def __repr__(self) -> str:
        return f"User(id={self.id!r}, name={self.bottleID!r}, fullname={self.filename!r})"
    
class CyverseInfo(Base):
    __tablename__ = 'cyverse'
    id = Column(Integer, primary_key=True, index=True)
    filename = Column(String)  
    def __repr__(self) -> str:
        return f"Address(id={self.id!r}, email_address={self.filename!r})"

In [ ]:
Base.metadata.create_all(engine)

In [ ]:
metadata_obj

In [ ]:
user_table_seqInfo = Table("sequencingInfo", metadata_obj, autoload_with=engine)

In [ ]:
#now insert data

In [ ]:
from sqlalchemy.orm import sessionmaker, Session
from datetime import datetime, time
from tqdm import tqdm

In [ ]:
Session = sessionmaker()

Session.configure(bind=engine)

In [ ]:
def load_sequencing_info():
    print('Loading sequencing information')
    data_dir = '../test_data/BIOS-SCOPE time series/'
    fName = 'V4_dada2_read_info_03052026.xlsx'
    df = pd.DataFrame(pd.read_excel(os.path.join(data_dir,fName)))
    #strip the @#%@#^$ spaces in headers
    df.columns = df.columns.str.replace(' ','') ## actual file has a space AFTER Bottle ID !

    #df = pd.read_csv('test_data/cruise_data.csv')
    session = Session()
    for index, row in df.iterrows():
        #pdb.set_trace()
        db_si = SeqInfo()
        db_si.bottleID = row['BottleID'] 
        db_si.cast = row['Cast']
        #db_si.filename = row['FilenameInCyverse']
        session.add(db_si)
    session.commit()

In [ ]:
load_sequencing_info()

In [ ]:
Session


In [ ]:
from sqlalchemy import inspect

session = inspect(Session).session

In [ ]:
type(Session)

In [ ]:
data_dir = '../test_data/BIOS-SCOPE time series/'

#careful the csv file and the xlsx have the same name but different information, I need the xlsx file
# fName = 'V4_dada2_read_info_03052026.csv'
#df_info = pd.DataFrame(pd.read_csv(os.path.join(data_dir,fName)))
fName = 'V4_dada2_read_info_03052026.xlsx'
df_info = pd.DataFrame(pd.read_excel(os.path.join(data_dir,fName)))

In [ ]:
#start with the sequence data, Luis gave me three lists. 
#Merge these and pull out relevant details, export to CSV file that will be read into the database
seqDir = '../test_data/Luis_fileLists'

dir_list = os.listdir(seqDir)
dir_list_full = [os.path.join(seqDir,f) for f in os.listdir(seqDir)] 